# Tool Calling

<br>

## What is Tool Calling?

Large Language Models are powerful at reasoning and generating text — but they have hard limits:

- They **can't access real-time data** (live weather, stock prices, news)
- They **can't run code** or perform precise calculations
- They **can't interact with external systems** (databases, APIs, files)

**Tool calling** (also called *function calling*) solves this. You give the model a list of tools it can request to use, and when the model decides a tool is needed, it tells *you* to run it and send the result back.

![tool-calling](../_images/tool-calling.jpg)


Note: the model never runs your code directly — it just requests it. You stay in control.


<br>

## How Tool Calling Works

The tool calling workflow typically has four steps:

1. User sends a message

2. Model decides whether a tool is needed
    - If so, it returns a request specifying which tool to call and its arguments
3. Your application executes the tool

4. You send the tool's result back to the model
    - The model uses it to generate the final response

This process can repeat multiple times if the model needs to call several tools before producing a final answer.


<br>

## Initial Setup

In [13]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

client = OpenAI()


<br>

## Step 1 — Define a Python Function and Its JSON Schema

To make a tool available to the model you need two things:

1. **The actual Python function** that will be executed when called
2. **A JSON schema** that describes the function to the model (name, description, parameters)

The model only sees the schema — it never sees your Python code.


In [14]:
import json

# The actual Python function (mocked — no real weather API needed)
def get_weather(city: str) -> str:
    mock_data = {
        "madrid":    {"temperature": "32°C", "condition": "Sunny"},
        "london":    {"temperature": "15°C", "condition": "Cloudy"},
        "new york":  {"temperature": "22°C", "condition": "Partly cloudy"},
        "tokyo":     {"temperature": "28°C", "condition": "Humid"},
    }
    data = mock_data.get(city.lower(), {"temperature": "20°C", "condition": "Unknown"})
    return f"Weather in {city}: {data['temperature']}, {data['condition']}"

print(get_weather("Madrid"))


Weather in Madrid: 32°C, Sunny


In [15]:
# The JSON schema that describes the function to the model


# Note: 
# - For this demo, we're defining a tool using OpenAI's tool schema.
# - Most LLM providers use a similar concept (name, description, and input schema), although the exact syntax and format may vary.

tools = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Returns the current weather for a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {
                    "type": "string",
                    "description": "The name of the city, e.g. 'Madrid' or 'Tokyo'."
                }
            },
            "required": ["city"],
            "additionalProperties": False
        }
    }
]

print("Tool schema defined ✓")


Tool schema defined ✓


<br>

## Step 2 — Call the Model with Tools Available

We pass the `tools` list to `responses.create()`. The model will decide on its own whether to answer directly or to request a tool call.


In [16]:
response = client.responses.create(
    model="gpt-5.4-nano",
    input="What's the weather like in Madrid right now?",
    tools=tools
)

# Inspect the raw output items
for item in response.output:
    print(f"type : {item.type}")
    print(f"item : {item}")
    print()


type : function_call
item : ResponseFunctionToolCall(arguments='{"city":"Madrid"}', call_id='call_STDa1dHUvRZCBwMOxjjVwvOA', name='get_weather', type='function_call', id='fc_0472328dd6d882ae006a572c79a1d081a0818d9ac9712b1b46', status='completed')



### What just happened?

The model didn't answer with text. Instead it returned a **`function_call`** item — it's saying:

> "I need to call `get_weather` with the argument `{"city": "Madrid"}`. Please run it and come back to me."

Let's extract the relevant fields:


In [17]:
# Find the function_call item in the response
tool_call = next(item for item in response.output if item.type == "function_call")

print("Function name :", tool_call.name)
print("Arguments     :", tool_call.arguments)   # JSON string
print("Call ID       :", tool_call.call_id)      # needed to return the result


Function name : get_weather
Arguments     : {"city":"Madrid"}
Call ID       : call_STDa1dHUvRZCBwMOxjjVwvOA


<br>

## Step 3 — Execute the Tool and Return the Result

Now we:
1. Parse the arguments (they come as a JSON string)
2. Call our Python function
3. Send the result back to the model in a second API call

The `input` list for the second call must include:
- The original user message
- The model's `function_call` output item (so the model remembers it asked)
- A `function_call_output` item with our result


In [18]:
user_message = "What's the weather like in Madrid right now?"

# 1. Parse the arguments
args = json.loads(tool_call.arguments)
print("Parsed args:", args)

# 2. Execute our function
result = get_weather(**args)
print("Tool result:", result)

# 3. Send everything back to the model
final_response = client.responses.create(
    model="gpt-5.4-nano",
    input=[
        {"role": "user", "content": user_message},   # original question
        tool_call,                                    # model's function_call item
        {                                             # our tool result
            "type": "function_call_output",
            "call_id": tool_call.call_id,
            "output": result
        }
    ],
    tools=tools
)

print("\nFinal answer:")
print(final_response.output_text)


Parsed args: {'city': 'Madrid'}
Tool result: Weather in Madrid: 32°C, Sunny

Final answer:
Right now in **Madrid** it’s **32°C** and **sunny**.


<br>

## Putting It All Together — The Agentic Loop

In production you won't manually wire each step. The standard pattern is a **loop** that keeps calling the model until it stops requesting tools and returns a final text answer.

```
while model requests a tool:
    execute the tool
    add result to conversation
    call model again
return model's text answer
```


In [19]:
# Registry: maps tool names to actual Python functions
TOOL_FUNCTIONS = {
    "get_weather": get_weather,
}

def run_with_tools(user_input: str, tools: list) -> str:
    """
    Calls the model in a loop, executing any requested tools,
    until the model produces a final text answer.
    """
    # Build the conversation as a list of input items
    messages = [{"role": "user", "content": user_input}]

    while True:
        response = client.responses.create(
            model="gpt-5.4-nano",
            input=messages,
            tools=tools
        )

        # Check if there are any tool calls in the response
        tool_calls = [item for item in response.output if item.type == "function_call"]

        if not tool_calls:
            # No more tool calls — return the final answer
            return response.output_text

        # Execute each requested tool and collect results
        for tool_call in tool_calls:
            fn = TOOL_FUNCTIONS[tool_call.name]
            args = json.loads(tool_call.arguments)
            result = fn(**args)
            print(f"  [tool] {tool_call.name}({args}) → {result}")

            # Append the model's function_call and our result to the conversation
            messages.append(tool_call)
            messages.append({
                "type": "function_call_output",
                "call_id": tool_call.call_id,
                "output": result
            })


In [20]:
answer = run_with_tools("What's the weather in London?", tools)
print("\nAnswer:", answer)


  [tool] get_weather({'city': 'London'}) → Weather in London: 15°C, Cloudy

Answer: The weather in **London** is **15°C** and **cloudy**.


<br>

## Multiple Tools — The Model Picks the Right One

You can give the model as many tools as you want. It will choose which one to call based on the user's question — or call none if it can answer directly.

Let's add a `get_stock_price` tool alongside `get_weather`.


In [21]:
# Mocked stock price tool — same pattern as get_weather
def get_stock_price(symbol: str) -> str:
    prices = {
        "aapl":  "$213.49",
        "googl": "$178.32",
        "msft":  "$415.60",
        "amzn":  "$195.88",
    }
    price = prices.get(symbol.lower(), "N/A")
    return f"{symbol.upper()}: {price}"

print(get_stock_price("AAPL"))
print(get_stock_price("AMZN"))


AAPL: $213.49
AMZN: $195.88


In [22]:
# Extended tools list
tools_v2 = [
    {
        "type": "function",
        "name": "get_weather",
        "description": "Returns the current weather for a given city.",
        "parameters": {
            "type": "object",
            "properties": {
                "city": {"type": "string", "description": "Name of the city"}
            },
            "required": ["city"],
            "additionalProperties": False
        }
    },
    {
        "type": "function",
        "name": "get_stock_price",
        "description": "Returns the current stock price for a given ticker symbol.",
        "parameters": {
            "type": "object",
            "properties": {
                "symbol": {"type": "string", "description": "Stock ticker symbol, e.g. 'AAPL' or 'AMZN'."}
            },
            "required": ["symbol"],
            "additionalProperties": False
        }
    }
]

# Update the registry
TOOL_FUNCTIONS["get_stock_price"] = get_stock_price

print("Two tools registered ✓")


Two tools registered ✓


In [24]:
# The model chooses get_weather
print("=== Weather question ===")
print(run_with_tools("What's the weather in Tokyo?", tools_v2))

print()

# The model chooses get_stock_price
print("=== Stock question ===")
print(run_with_tools("What is the current price of Google stock?", tools_v2))

print()

# The model answers directly — no tool needed
print("=== Direct question ===")
print(run_with_tools("What is the capital of France?", tools_v2))


=== Weather question ===
  [tool] get_weather({'city': 'Tokyo'}) → Weather in Tokyo: 28°C, Humid
In **Tokyo**, it’s currently **28°C** and **humid**.

=== Stock question ===
  [tool] get_stock_price({'symbol': 'GOOGL'}) → GOOGL: $178.32
Google stock (GOOGL) is currently **$178.32**.

=== Direct question ===
The capital of France is **Paris**.


<br>

---

<br>

## Provider-Managed Tools (e.g., `web_search`)

Everything we've built so far are **custom tools**: you define them, you execute them, you return the result.

Some AI providers also offer **built-in tools** that they run entirely on your behalf. OpenAI's `web_search` is a prime example — you just enable it, and the model searches the web automatically when it decides it's needed. No function to write, no result to return, no loop to manage.

| | Custom tools | Provider-managed tools |
|---|---|---|
| You define the schema | ✅ | ❌ |
| You execute the function | ✅ | ❌ |
| Agentic loop needed | ✅ | ❌ |
| Example | `get_weather`, `get_stock_price` | `web_search` |


<br>

### Using `web_search` with the Responses API

Enable it by adding `{"type": "web_search_preview"}` to the `tools` list. That's it — the model decides when to search and handles everything internally.


In [31]:
# Question that requires up-to-date information
response = client.responses.create(
    model="gpt-5.4-nano",
    tools=[{"type": "web_search_preview"}],
    input="What are the top AI news stories this week?"
)

print(response.output_text)

Here are some of the **top AI news stories from this week (mid–July 2026)**, based on the latest reporting I can find:

1) **Google DeepMind leader Demis Hassabis proposes a U.S.-led global AI watchdog**
   - Axios reports Hassabis argues for a more “systematic” approach to AI regulation, including an industry-funded, technically staffed standards body accountable to the U.S. government. ([axios.com](https://www.axios.com/2026/07/14/demis-hassabis-ai-regulation-google-deepmind?utm_source=openai))

2) **Meta lays out an AI revenue plan (charging developers for model access)**
   - Axios highlights Meta’s shifting strategy to monetize AI—e.g., updates to its “Muse Spark” model with paid access for developers, plus broader push into subscriptions/enterprise access. ([axios.com](https://www.axios.com/newsletters/axios-closer-41148026-0f2b-4191-b48f-84cd7859c9d0?utm_source=openai))

3) **AI-focused cybersecurity: “hackers embrace AI” trend**
   - An Axios newsletter notes reporting about in

The model searched the web on its own — no tool execution code on our side.

<br>

Notice that for a question the model can answer from its training data, it skips the search entirely:


In [32]:
# Question the model can answer from its own knowledge — no search triggered
response = client.responses.create(
    model="gpt-5.4-nano",
    tools=[{"type": "web_search_preview"}],
    input="What year was the Eiffel Tower built? (answer briefly)"
)

# Check whether the model used web search or answered directly
for item in response.output:
    print(f"type: {item.type}")

print()
print(response.output_text)


type: message

The Eiffel Tower was built in **1889**.
